# 入力画像 EDA

アノテーションデータと入力画像を探索的分析し、以下のパラメータ選定の根拠を得る:
- `patch_t_min` / `patch_t_max` — パッチ抽出のt範囲
- `patch_perp_scale` — 垂直方向カバレッジ
- `label_t_min` / `label_t_max` — ラベル正規化範囲

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# プロジェクトルートをパスに追加
ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import math
import random
from collections import defaultdict

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from eyeliner.data.dataset import load_manifest
from utils.patch_utils import extract_patch, T_MIN, T_MAX, PATCH_SIZE

print("ROOT:", ROOT)

## 1. データ読み込み

In [ ]:
MANIFEST_PATH = ROOT / "experiments/EXP006_data_increase_sweep/manifests/manifest_train.csv"

records = load_manifest(MANIFEST_PATH)


def compute_meta(records: list[dict]) -> None:
    """compute_eye_meta の拡張版: pq_dist / eye_scale も records に追加する（in-place）。"""
    eye_points: dict = defaultdict(list)
    eye_count: dict = defaultdict(int)
    for rec in records:
        key = (rec["image_path"], rec["part_name"])
        eye_points[key].append((float(rec["px"]), float(rec["py"])))
        eye_points[key].append((float(rec["qx"]), float(rec["qy"])))
        eye_count[key] += 1

    eye_scale_map: dict = {}
    for key, pts in eye_points.items():
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        scale = math.sqrt((max(xs) - min(xs)) ** 2 + (max(ys) - min(ys)) ** 2)
        eye_scale_map[key] = max(scale, 1.0)

    for rec in records:
        key = (rec["image_path"], rec["part_name"])
        pq_dist = math.sqrt(
            (float(rec["qx"]) - float(rec["px"])) ** 2
            + (float(rec["qy"]) - float(rec["py"])) ** 2
        )
        eye_scale = eye_scale_map[key]
        n_pq = eye_count[key]
        rec["pq_dist"] = pq_dist
        rec["eye_scale"] = eye_scale
        rec["pq_ratio"] = pq_dist / eye_scale
        rec["weight"] = pq_dist / (eye_scale * n_pq)
        rec["val_weight"] = pq_dist / n_pq


compute_meta(records)

df = pd.DataFrame(records)
for col in ["t", "px", "py", "qx", "qy", "pq_dist", "eye_scale", "pq_ratio"]:
    df[col] = df[col].astype(float)
df["point_index"] = df["point_index"].astype(int)

print(f"サンプル数: {len(df):,}")
print(f"ユニーク画像数: {df['image_path'].nunique():,}")
print(df.head(3))

## 2. 基本統計

In [ ]:
print("=== サンプル内訳 ===")
print(df.groupby(["part_name", "sub_key"]).size().to_string())
print()

print("=== ユニーク目数 ===")
n_eyes = df.groupby(["image_path", "part_name"]).ngroups
print(f"ユニーク目: {n_eyes}")
print()

print("=== t値の基本統計 ===")
pct = [1, 5, 10, 25, 50, 75, 90, 95, 99]
print(df["t"].describe().to_string())
print()
print("パーセンタイル:")
for p in pct:
    print(f"  p{p:02d}: {df['t'].quantile(p/100):.4f}")

## 3. t値の分布分析

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- 全体のt分布 ---
axes[0].hist(df["t"], bins=80, color="steelblue", edgecolor="white", linewidth=0.3)
for tval, color, label in [
    (-0.5, "orange", "t_min=-0.5"),
    (1.5,  "orange", "t_max=1.5"),
    (-1.0, "red",    "t_min=-1.0"),
    (2.0,  "red",    "t_max=2.0"),
    (0.0,  "gray",   "Q (t=0)"),
    (1.0,  "gray",   "P (t=1)"),
]:
    axes[0].axvline(tval, color=color, linestyle="--", linewidth=1.2, alpha=0.8)
axes[0].set_title("t値の分布（全体）")
axes[0].set_xlabel("t")
axes[0].set_ylabel("count")

# --- upper / lower 別 ---
for sub_key, color in [("upper", "steelblue"), ("lower", "tomato")]:
    axes[1].hist(df[df["sub_key"] == sub_key]["t"], bins=60,
                 alpha=0.6, color=color, label=sub_key, edgecolor="white", linewidth=0.3)
axes[1].axvline(0.0, color="gray", linestyle="--", linewidth=1)
axes[1].axvline(1.0, color="gray", linestyle="--", linewidth=1)
axes[1].legend()
axes[1].set_title("t値の分布（upper / lower）")
axes[1].set_xlabel("t")

plt.tight_layout()
plt.show()

# --- クランプ影響の定量評価 ---
print("=== patch_t設定ごとのクランプ影響 ===")
for tmin, tmax, name in [
    (-0.5, 1.5, "I1 baseline (-0.5 / 1.5)"),
    (-1.0, 2.0, "I2 wide_patch (-1.0 / 2.0)"),
    (-0.5, 1.5, "label_t (-0.5 / 1.5) ← I2のlabel設定"),
]:
    out_of_range = ((df["t"] < tmin) | (df["t"] > tmax)).sum()
    pct_out = out_of_range / len(df) * 100
    print(f"  {name}: 範囲外 {out_of_range:,} 件 ({pct_out:.2f}%)")

## 4. スケール関連分析

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# --- pq_dist ---
axes[0, 0].hist(df["pq_dist"], bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0, 0].set_title("pq_dist (P→Q距離) [px]")
axes[0, 0].set_xlabel("pixels")

# --- eye_scale ---
axes[0, 1].hist(df["eye_scale"], bins=60, color="darkorange", edgecolor="white", linewidth=0.3)
axes[0, 1].set_title("eye_scale (目バウンディング対角線) [px]")
axes[0, 1].set_xlabel("pixels")

# --- pq_ratio ---
axes[0, 2].hist(df["pq_ratio"], bins=60, color="seagreen", edgecolor="white", linewidth=0.3)
axes[0, 2].set_title("pq_ratio = pq_dist / eye_scale")
axes[0, 2].set_xlabel("ratio")

# --- pq_dist vs eye_scale (scatter) ---
axes[1, 0].scatter(df["eye_scale"], df["pq_dist"],
                   c=df["pq_ratio"], cmap="viridis", s=2, alpha=0.3)
axes[1, 0].set_xlabel("eye_scale [px]")
axes[1, 0].set_ylabel("pq_dist [px]")
axes[1, 0].set_title("pq_dist vs eye_scale (色=pq_ratio)")

# --- t vs pq_ratio ---
axes[1, 1].scatter(df["t"], df["pq_ratio"],
                   c=df["sub_key"].map({"upper": 0, "lower": 1}),
                   cmap="bwr", s=2, alpha=0.3)
axes[1, 1].set_xlabel("t")
axes[1, 1].set_ylabel("pq_ratio")
axes[1, 1].set_title("t vs pq_ratio (青=upper, 赤=lower)")

# --- pq_dist (pixel) の累積分布 ---
sorted_pq = np.sort(df["pq_dist"].values)
cdf = np.arange(1, len(sorted_pq) + 1) / len(sorted_pq)
axes[1, 2].plot(sorted_pq, cdf, color="steelblue")
for p_val in [0.1, 0.25, 0.5, 0.75, 0.9]:
    v = np.quantile(sorted_pq, p_val)
    axes[1, 2].axvline(v, color="gray", linestyle=":", linewidth=0.8)
    axes[1, 2].text(v, p_val, f" p{int(p_val*100)}={v:.0f}px", fontsize=7)
axes[1, 2].set_xlabel("pq_dist [px]")
axes[1, 2].set_ylabel("CDF")
axes[1, 2].set_title("pq_dist の累積分布")

plt.tight_layout()
plt.show()

print("=== スケール統計サマリー ===")
print(df[["pq_dist", "eye_scale", "pq_ratio"]].describe().round(2).to_string())

## 5. 画像サイズ分析

In [ ]:
# 画像サイズを取得（ユニーク画像のみ）
unique_images = df["image_path"].unique()
img_sizes = []
for path in unique_images:
    full_path = ROOT / path
    img = cv2.imread(str(full_path))
    if img is not None:
        h, w = img.shape[:2]
        img_sizes.append({"image_path": path, "width": w, "height": h})

df_img = pd.DataFrame(img_sizes)
df_img["aspect"] = df_img["width"] / df_img["height"]

print(f"読み込み成功: {len(df_img)} / {len(unique_images)} 枚")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_img["width"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("画像 width [px]")
axes[0].set_xlabel("pixels")

axes[1].hist(df_img["height"], bins=40, color="darkorange", edgecolor="white")
axes[1].set_title("画像 height [px]")
axes[1].set_xlabel("pixels")

axes[2].scatter(df_img["width"], df_img["height"], s=5, alpha=0.4, color="seagreen")
axes[2].set_xlabel("width")
axes[2].set_ylabel("height")
axes[2].set_title("width vs height")

plt.tight_layout()
plt.show()

print(df_img[["width", "height", "aspect"]].describe().round(1).to_string())

## 6. インタラクティブ パッチビジュアライザ

**操作方法:**
- `idx` — manifest の行インデックス（IntText で直接入力可）
- `patch_t_min / max` — パッチのt抽出範囲
- `patch_perp_scale` — 垂直カバレッジ（1.0 = PQ長と同じ幅）
- `patch_h / w` — パッチ解像度
- **Random** ボタン群 — フィルタ付きランダムジャンプ

In [ ]:
# 画像キャッシュ（遅延ロード）
_image_cache: dict[str, np.ndarray] = {}


def load_image(image_path: str) -> np.ndarray | None:
    if image_path not in _image_cache:
        full = ROOT / image_path
        bgr = cv2.imread(str(full))
        if bgr is not None:
            _image_cache[image_path] = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        else:
            _image_cache[image_path] = None
    return _image_cache[image_path]


def patch_corners_in_image(
    px: float, py: float, qx: float, qy: float,
    patch_t_min: float, patch_t_max: float,
    patch_perp_scale: float, patch_h: int, patch_w: int,
) -> list[tuple[float, float]] | None:
    """パッチの4隅（左上→右上→右下→左下）の画像座標を返す。"""
    dx, dy = qx - px, qy - py
    norm = float(np.hypot(dx, dy))
    if norm == 0:
        return None
    ux, uy = dx / norm, dy / norm
    sx = patch_t_max * px + (1 - patch_t_max) * qx
    sy = patch_t_max * py + (1 - patch_t_max) * qy

    corners = []
    for (pc_x, pc_y) in [(0, 0), (patch_w - 1, 0), (patch_w - 1, patch_h - 1), (0, patch_h - 1)]:
        s_para = pc_x * (patch_t_max - patch_t_min) * norm / (patch_w - 1)
        d = (pc_y - patch_h // 2) * patch_perp_scale * norm / (patch_h - 1)
        img_x = sx + s_para * ux + d * (-uy)
        img_y = sy + s_para * uy + d * ux
        corners.append((img_x, img_y))
    return corners


def t_to_patch_x(t: float, patch_t_min: float, patch_t_max: float, patch_w: int) -> float:
    """t値をパッチ上のx座標に変換する。"""
    return (patch_t_max - t) / (patch_t_max - patch_t_min) * (patch_w - 1)

In [ ]:
# ---- Widgets 定義 ----

w_idx = widgets.BoundedIntText(
    value=0, min=0, max=len(df) - 1,
    description="idx:", layout=widgets.Layout(width="200px")
)
w_t_min = widgets.FloatSlider(
    value=-0.5, min=-3.0, max=0.0, step=0.05,
    description="t_min:", readout_format=".2f",
    style={"description_width": "60px"}, layout=widgets.Layout(width="350px")
)
w_t_max = widgets.FloatSlider(
    value=1.5, min=1.0, max=4.0, step=0.05,
    description="t_max:", readout_format=".2f",
    style={"description_width": "60px"}, layout=widgets.Layout(width="350px")
)
w_perp = widgets.FloatSlider(
    value=1.0, min=0.1, max=4.0, step=0.1,
    description="perp_scale:", readout_format=".1f",
    style={"description_width": "80px"}, layout=widgets.Layout(width="350px")
)
w_h = widgets.IntSlider(
    value=64, min=16, max=256, step=8,
    description="patch_h:",
    style={"description_width": "70px"}, layout=widgets.Layout(width="350px")
)
w_w = widgets.IntSlider(
    value=128, min=32, max=512, step=8,
    description="patch_w:",
    style={"description_width": "70px"}, layout=widgets.Layout(width="350px")
)

# Random ボタン群
btn_random_all   = widgets.Button(description="Random (全体)",       button_style="info")
btn_random_upper = widgets.Button(description="Random (upper)",      button_style="")
btn_random_lower = widgets.Button(description="Random (lower)",      button_style="")
btn_random_edge  = widgets.Button(description="Random (端点 t<0.1 or >0.9)", button_style="warning")
btn_random_small = widgets.Button(description="Random (小eye_scale)", button_style="danger")
btn_random_large = widgets.Button(description="Random (大eye_scale)", button_style="success")

out = widgets.Output()


def pick_random(mask=None):
    subset = df[mask] if mask is not None else df
    if len(subset) == 0:
        return
    w_idx.value = int(subset.sample(1).index[0])


q20 = df["eye_scale"].quantile(0.20)
q80 = df["eye_scale"].quantile(0.80)

btn_random_all.on_click(lambda _: pick_random())
btn_random_upper.on_click(lambda _: pick_random(df["sub_key"] == "upper"))
btn_random_lower.on_click(lambda _: pick_random(df["sub_key"] == "lower"))
btn_random_edge.on_click(lambda _: pick_random((df["t"] < 0.1) | (df["t"] > 0.9)))
btn_random_small.on_click(lambda _: pick_random(df["eye_scale"] < q20))
btn_random_large.on_click(lambda _: pick_random(df["eye_scale"] > q80))

print(f"小eye_scale閾値 (p20): {q20:.1f}px")
print(f"大eye_scale閾値 (p80): {q80:.1f}px")

In [ ]:
def render(idx, patch_t_min, patch_t_max, patch_perp_scale, patch_h, patch_w):
    if idx >= len(df):
        print(f"idx {idx} はデータ範囲外")
        return

    rec = df.iloc[idx]
    px, py = float(rec["px"]), float(rec["py"])
    qx, qy = float(rec["qx"]), float(rec["qy"])
    t_actual = float(rec["t"])
    pq_dist  = float(rec["pq_dist"])
    eye_scale = float(rec["eye_scale"])
    pq_ratio = float(rec["pq_ratio"])

    img = load_image(rec["image_path"])
    if img is None:
        print(f"画像が見つかりません: {rec['image_path']}")
        return

    # パッチ抽出
    patch = extract_patch(
        img, px, py, qx, qy,
        patch_size=(patch_h, patch_w),
        t_min=patch_t_min, t_max=patch_t_max,
        perp_scale=patch_perp_scale,
    )

    corners = patch_corners_in_image(
        px, py, qx, qy,
        patch_t_min, patch_t_max, patch_perp_scale, patch_h, patch_w,
    )

    # 元画像: 周囲にマージンを取ってクロップ
    margin = max(80, int(eye_scale * 0.5))
    all_x = [px, qx] + ([c[0] for c in corners] if corners else [])
    all_y = [py, qy] + ([c[1] for c in corners] if corners else [])
    x1 = max(0, int(min(all_x)) - margin)
    y1 = max(0, int(min(all_y)) - margin)
    x2 = min(img.shape[1], int(max(all_x)) + margin)
    y2 = min(img.shape[0], int(max(all_y)) + margin)
    cropped = img[y1:y2, x1:x2]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- 左: 元画像 + オーバーレイ ---
    axes[0].imshow(cropped)

    if corners:
        poly_pts = np.array([(c[0] - x1, c[1] - y1) for c in corners])
        polygon = plt.Polygon(poly_pts, closed=True, fill=False,
                              edgecolor="yellow", linewidth=2, linestyle="--")
        axes[0].add_patch(polygon)

    axes[0].plot(px - x1, py - y1, "bo", markersize=9, label="P (t=1)", zorder=5)
    axes[0].plot(qx - x1, qy - y1, "ro", markersize=9, label="Q (t=0)", zorder=5)
    axes[0].plot([px - x1, qx - x1], [py - y1, qy - y1], "g-", linewidth=1.5, alpha=0.7)

    rx = t_actual * px + (1 - t_actual) * qx
    ry = t_actual * py + (1 - t_actual) * qy
    axes[0].plot(rx - x1, ry - y1, "g*", markersize=12, label=f"t={t_actual:.3f}", zorder=5)

    axes[0].set_title(
        f"元画像  idx={idx} | {rec['part_name']} {rec['sub_key']}"
    )
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].axis("off")

    # --- 右: 抽出パッチ ---
    patch_u8 = (patch * 255).clip(0, 255).astype(np.uint8)
    axes[1].imshow(patch_u8, aspect="auto")

    def _vline(t_val, color, label):
        x = t_to_patch_x(t_val, patch_t_min, patch_t_max, patch_w)
        if 0 <= x <= patch_w:
            axes[1].axvline(x, color=color, linewidth=2, linestyle="--", label=label)

    _vline(1.0, "blue",  "P (t=1)")
    _vline(0.0, "red",   "Q (t=0)")
    _vline(t_actual, "lime", f"t={t_actual:.3f}")

    axes[1].set_title(
        f"抽出パッチ ({patch_h}×{patch_w})\n"
        f"t_min={patch_t_min:.2f}  t_max={patch_t_max:.2f}  perp={patch_perp_scale:.1f}"
    )
    axes[1].legend(loc="upper right", fontsize=8)
    axes[1].axis("off")

    # メタ情報
    meta = (
        f"t={t_actual:.4f}  |  eye_scale={eye_scale:.1f}px  |  "
        f"pq_dist={pq_dist:.1f}px  |  pq_ratio={pq_ratio:.3f}  |  "
        f"img={img.shape[1]}×{img.shape[0]}  |  {rec['image_path']}"
    )
    fig.text(0.5, 0.01, meta, ha="center", fontsize=8,
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

    plt.tight_layout(rect=[0, 0.06, 1, 1])
    plt.show()


interactive_out = widgets.interactive_output(
    render,
    dict(
        idx=w_idx,
        patch_t_min=w_t_min,
        patch_t_max=w_t_max,
        patch_perp_scale=w_perp,
        patch_h=w_h,
        patch_w=w_w,
    ),
)

ui = widgets.VBox([
    widgets.HBox([
        w_idx,
        btn_random_all, btn_random_upper, btn_random_lower,
    ]),
    widgets.HBox([btn_random_edge, btn_random_small, btn_random_large]),
    widgets.HBox([w_t_min, w_t_max]),
    widgets.HBox([w_perp]),
    widgets.HBox([w_h, w_w]),
])

display(ui, interactive_out)